# TGLC 200s Sanity Check

Exploratory notebook verifying that TGLC (Sector 56+, 200s cadence) is a viable replacement for SPOC 2-min as the primary light curve source.
Checks availability, cadence, light curve quality, and window re-parameterization.

---
## Section 0: Imports & Setup

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
import time
import threading
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from astroquery.mast import Catalogs, Observations
import lightkurve as lk

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

# astroquery socket-level timeout
Observations.TIMEOUT = 20

# --- Project root detection (walk up until CLAUDE.md is found) ---
project_root = r'c:\git_repo\Stellar-World-Model'

print(f'Project root: {project_root}')

Project root: c:\git_repo\Stellar-World-Model


---
## Section 1: Query Bright Nearby Stars from TIC v8 (TESS Input Catalog)

Query MAST TIC v8 directly with `Tmag < 7`, `plx > 10 mas`, `objType = STAR`.
Randomly sample 20 stars from the result for the TGLC availability check.

In [7]:
import time

Catalogs.TIMEOUT = 300

def _query_tic_with_retry(criteria, max_retries=4, base_wait=8):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            print(f'  MAST attempt {attempt}/{max_retries}...')
            return Catalogs.query_criteria(catalog='TIC', **criteria)
        except Exception as e:
            last_err = e
            print(f'    {type(e).__name__}: {str(e)[:150]}')
            if attempt < max_retries:
                wait = base_wait * attempt
                print(f'    Retrying in {wait}s...')
                time.sleep(wait)
    raise last_err

print('Querying TIC v8 from MAST (Tmag < 7, plx > 10 mas)...')
try:
    catalog = _query_tic_with_retry(dict(Tmag=[-99, 7], plx=[10, 9999], objType='STAR'))
except Exception as e:
    print(f'Query failed after retries ({type(e).__name__}).')
    print('Falling back to tighter parallax cut (plx > 20 mas) to shrink payload...')
    catalog = _query_tic_with_retry(dict(Tmag=[-99, 7], plx=[20, 9999], objType='STAR'))

print(f'MAST returned {len(catalog)} rows.')

df_tic = catalog.to_pandas()
for col in df_tic.columns:
    df_tic[col] = pd.to_numeric(df_tic[col], errors='coerce')

col_lower = {c.lower(): c for c in df_tic.columns}

def _get_col(*candidates):
    for name in candidates:
        found = col_lower.get(name.lower())
        if found:
            return found
    return None

bp_col = _get_col('gaiaBP', 'gaiaBp', 'gaia_bp')
rp_col = _get_col('gaiaRP', 'gaiaRp', 'gaia_rp')
g_col  = _get_col('GAIAmag', 'gaiamag', 'gaia_g')
print(f'Gaia columns found -> G: {g_col},  BP: {bp_col},  RP: {rp_col}')

required_cols = ['ID', 'Tmag', 'Teff', 'logg', 'MH', 'plx', 'rad', 'mass']
extra_cols = [c for c in [g_col, bp_col, rp_col] if c is not None]
df_tic = df_tic[required_cols + extra_cols].copy()

rename = {}
if bp_col and bp_col != 'gaiaBp':  rename[bp_col] = 'gaiaBp'
if rp_col and rp_col != 'gaiaRp':  rename[rp_col] = 'gaiaRp'
if g_col  and g_col  != 'GAIAmag': rename[g_col]  = 'GAIAmag'
if rename:
    df_tic = df_tic.rename(columns=rename)

print(f'Stars after TIC query (Tmag < 7, plx > 10 mas): {len(df_tic)}')
print()
print(df_tic.head(5).to_string(index=False))

sample20 = df_tic.sample(n=20, random_state=42).reset_index(drop=True)
tic_ids_20 = sample20['ID'].astype(int).tolist()

print(f'20 randomly sampled TIC IDs (random_state=42):')
print(tic_ids_20)

Querying TIC v8 from MAST (Tmag < 7, plx > 10 mas)...
  MAST attempt 1/4...
MAST returned 7121 rows.
Gaia columns found -> G: GAIAmag,  BP: gaiabp,  RP: gaiarp
Stars after TIC query (Tmag < 7, plx > 10 mas): 7121

       ID   Tmag   Teff    logg   MH     plx      rad  mass  GAIAmag  gaiaBp  gaiaRp
117927293 5.9418 6384.0 3.81453  NaN 16.5095 2.319380  1.28  6.30145 6.58443 5.90321
238432056 5.3971 5489.0 4.52192 0.07 72.5764 0.889627  0.96  5.87245 6.26563 5.35394
395353076 6.4809 5846.0 3.94594  NaN 19.2461 1.814320  1.06  6.90115 7.24737 6.44759
405190745 5.7287 7155.0 3.78028  NaN 13.3128 2.697460  1.60  5.95194 6.12404 5.69549
293305996 6.8766 7186.0 4.26338  NaN 13.4562 1.551510  1.61  7.11138 7.29594 6.84684
20 randomly sampled TIC IDs (random_state=42):
[153498031, 336893395, 1307739862, 351354256, 67265166, 94834355, 406500231, 243995474, 264595454, 425312220, 329759640, 52829336, 335870780, 71450103, 99820700, 441038084, 202380743, 468979699, 351535191, 117297589]


---
## Section 2: TGLC 200s Availability Check

For each of the 20 sampled TIC IDs, query MAST for TGLC light curves and filter to Sector 56+ (200s cadence, Cycles 5+).
Uses the same `search_with_timeout` + MAST session reset pattern as the reference notebook.

In [12]:
# --- Constants ---
QUERY_TIMEOUT  = 20    # astroquery socket-level cap (seconds)
THREAD_TIMEOUT = 25    # daemon thread cap (fires only if astroquery's own timeout misses)
QUERY_DELAY    = 0.5   # politeness sleep between queries
MIN_SECTOR     = 56    # first sector with 200s FFI cadence


def _reset_mast_session():
    """Drop astroquery's HTTP session so dangling connections are closed."""
    try:
        sess = getattr(Observations, '_session', None)
        if sess is not None:
            sess.close()
    except Exception:
        pass
    try:
        Observations._session = requests.Session()
    except Exception:
        pass


def search_tglc_with_timeout(tic_id: int, timeout: int = THREAD_TIMEOUT):
    """Return (n_sectors_56plus, sector_list_56plus, status).
    status in {ok, timeout, error:<typename>}.
    """
    result = [0, []]
    exc = [None]

    def _run():
        try:
            sr = lk.search_lightcurve(f'TIC {tic_id}', mission='TESS', author='TGLC')
            if len(sr) == 0:
                result[0], result[1] = 0, []
                return
            try:
                all_sectors = list(sr.table['sequence_number'])
            except Exception:
                all_sectors = []
            sectors_56plus = [s for s in all_sectors if int(s) >= MIN_SECTOR]
            result[0] = len(sectors_56plus)
            result[1] = sectors_56plus
        except Exception as e:
            exc[0] = e

    t = threading.Thread(target=_run, daemon=True)
    t.start()
    t.join(timeout)

    if t.is_alive():
        return 0, [], 'timeout'
    if exc[0] is not None:
        name = type(exc[0]).__name__
        if 'Timeout' in name or 'ConnectionError' in name:
            return 0, [], 'timeout'
        return 0, [], f'error:{name}'
    return result[0], result[1], 'ok'

In [13]:
# --- MAST warmup sanity check ---
WARMUP_TIC = tic_ids_20[0]
print(f'MAST warmup on TIC {WARMUP_TIC}...')
t_warm = time.time()
n_warm, sec_warm, status_warm = search_tglc_with_timeout(WARMUP_TIC)
dt_warm = time.time() - t_warm
print(f'  status={status_warm}  n_sectors_56plus={n_warm}  took {dt_warm:.1f}s')
if status_warm not in ('ok',):
    raise RuntimeError(f'MAST warmup failed: {status_warm}. Check https://outerspace.stsci.edu/display/MASTDOCS')
if dt_warm > 5:
    print(f'  Warning: MAST is responding but slowly ({dt_warm:.1f}s). Loop will be slow.')

# --- Main loop over 20-star sample ---
print(f'\nQuerying TGLC availability for 20 stars (Sector >= {MIN_SECTOR})...')
print(f'{"-"*70}')

avail_records = []
consec_bad = 0
t0 = time.time()

for i, tic_id in enumerate(tic_ids_20, start=1):
    t_q = time.time()
    n, sectors, status = search_tglc_with_timeout(tic_id)
    dt = time.time() - t_q

    if status == 'ok':
        consec_bad = 0
        avail_records.append({
            'tic_id': tic_id,
            'n_tglc_sectors_56plus': n,
            'sector_list_56plus': sectors,
        })
        print(f'  [{i:>2}/20]  TIC {tic_id:>11}  n_56plus={n:>3}  {dt:>5.1f}s  sectors={sectors}')
    else:
        consec_bad += 1
        # Do NOT append — unknown answer; re-run cell to retry this star
        if status == 'timeout':
            _reset_mast_session()
            backoff = min(5 * (2 ** min(consec_bad - 1, 4)), 60)
            print(f'  [{i:>2}/20]  TIC {tic_id:>11}  TIMEOUT — reset session, back off {backoff}s')
            time.sleep(backoff)
        else:
            print(f'  [{i:>2}/20]  TIC {tic_id:>11}  {status}')

    if i < 20:
        time.sleep(QUERY_DELAY)

print(f'{"-"*70}')
print(f'Done in {time.time() - t0:.1f}s  ({len(avail_records)}/20 queries recorded)')

df_avail = pd.DataFrame(avail_records)

MAST warmup on TIC 153498031...
  status=ok  n_sectors_56plus=0  took 0.0s

Querying TGLC availability for 20 stars (Sector >= 56)...
----------------------------------------------------------------------
  [ 1/20]  TIC   153498031  n_56plus=  0    0.0s  sectors=[]
  [ 2/20]  TIC   336893395  n_56plus=  0    0.0s  sectors=[]
  [ 3/20]  TIC  1307739862  n_56plus=  0    0.0s  sectors=[]
  [ 4/20]  TIC   351354256  n_56plus=  0    0.0s  sectors=[]
  [ 5/20]  TIC    67265166  n_56plus=  0    0.0s  sectors=[]
  [ 6/20]  TIC    94834355  n_56plus=  0    0.0s  sectors=[]
  [ 7/20]  TIC   406500231  n_56plus=  0    0.0s  sectors=[]
  [ 8/20]  TIC   243995474  n_56plus=  0    0.0s  sectors=[]
  [ 9/20]  TIC   264595454  n_56plus=  0    0.0s  sectors=[]
  [10/20]  TIC   425312220  n_56plus=  0    0.0s  sectors=[]
  [11/20]  TIC   329759640  n_56plus=  0    0.0s  sectors=[]


No data found for target "TIC 52829336".


  [12/20]  TIC    52829336  n_56plus=  0    5.9s  sectors=[]


No data found for target "TIC 335870780".


  [13/20]  TIC   335870780  n_56plus=  0   14.3s  sectors=[]
  [14/20]  TIC    71450103  TIMEOUT — reset session, back off 5s


...

  [19/20]  TIC   351535191  TIMEOUT — reset session, back off 5s


No data found for target "TIC 351535191".


  [20/20]  TIC   117297589  TIMEOUT — reset session, back off 10s


No data found for target "TIC 117297589".


----------------------------------------------------------------------
Done in 181.7s  (16/20 queries recorded)
